# Publish PO Price Anomaly Scores to SAP BDC

This notebook publishes the `PO_PRICE_ANOMALY_SCORES` Iceberg table from `TECHED_DA162.SCHEMA_NP` as a data product to SAP BDC via the zerocopy connector.

**Steps:**
1. Set role and warehouse context
2. Create share and grant privileges
3. Attach share to the SAP BDC zerocopy connector
4. Publish data product with CSN metadata

In [ ]:
%%sql -r ctx_result
USE WAREHOUSE SYSTEM$STREAMLIT_NOTEBOOK_WH;

✍️ In the share `TECHED_SHARE_<USERNAME>`, replace your username. Replace <SCHEMANAME> with your SCHEMA_USERNAME

In [ ]:
%%sql -r share_result
CREATE SHARE IF NOT EXISTS TECHED_SHARE_<USERNAME>
  COMMENT = 'Share for PO Price Anomaly Scores data product to SAP BDC';

GRANT USAGE ON DATABASE TECHED_DA162 TO SHARE TECHED_SHARE_<USERNAME>;
GRANT USAGE ON SCHEMA TECHED_DA162.<SCHEMANAME> TO SHARE TECHED_SHARE_<USERNAME>;
GRANT SELECT ON TABLE TECHED_DA162.<SCHEMANAME>.PO_PRICE_ANOMALY_SCORES TO SHARE TECHED_SHARE_<USERNAME>;

✍️ In the share `TECHED_SHARE_<USERNAME>`, replace your username. 

In [ ]:
%%sql -r attach_result
BEGIN
    ALTER ZEROCOPY CONNECTOR DB_BDC_SANDBOX.PUBLIC.SAP_BDC_CONNECT_SANDBOX_BFSNOWFLAKE
      ADD SHARE TECHED_SHARE_<USERNAME>;
EXCEPTION
    WHEN OTHER THEN
        -- Share may already be associated with the connector
        RETURN 'Share TECHED_SHARE_<USERNAME> is already associated with the connector (or another error occurred: ' || SQLERRM || ')';
END;

✍️ In the share `TECHED_SHARE_<USERNAME>`, replace your username. In the title, suffix the data product name with your username.

In [ ]:
SELECT SYSTEM$SAP_PUBLISH_DATA_PRODUCT(
  'DB_BDC_SANDBOX.PUBLIC.SAP_BDC_CONNECT_SANDBOX_BFSNOWFLAKE',
  'TECHED_SHARE_<USERNAME>',
  $$
{
  "title": "<USERNAME> Purchase Order Anomaly Scores",
  "shortDescription": "PO price anomaly scores with overpriced and underpriced flags.",
  "description": "Enriched purchase order data with ML-driven price anomaly detection. Each record includes composite anomaly scores derived from z-score analysis across net price, order quantity, and order value dimensions. Records are classified as NORMAL, OVERPRICED, or UNDERPRICED with confidence levels ranging from 30 to 100."
}
$$,
  $$
{
  "csnInteropEffective": "1.0",
  "$version": "2.0",
  "i18n": {},
  "meta": {
    "creator": "Snowflake CSN Interop Generator - Minimal",
    "flavor": "inferred"
  },
  "definitions": {
    "<SCHEMANAME>": { "kind": "context" },
    "<SCHEMANAME>.<TABLENAME>": {
      "kind": "entity",
      "elements": {
        "PURCHASING_DOCUMENT": {
          "type": "cds.String",
          "key": true,
          "notNull": true
        },
        "PURCHASING_DOCUMENT_ITEM": {
          "type": "cds.String",
          "key": true,
          "notNull": true
        },
        "MATERIAL": {
          "type": "cds.String"
        },
        "MATERIAL_GROUP": {
          "type": "cds.String"
        },
        "SUPPLIER": {
          "type": "cds.String"
        },
        "PLANT": {
          "type": "cds.String"
        },
        "COMPANY_CODE": {
          "type": "cds.String"
        },
        "DOCUMENT_DATE": {
          "type": "cds.Date"
        },
        "NET_PRICE": {
          "type": "cds.Decimal",
          "precision": 34,
          "scale": 4
        },
        "ORDER_QUANTITY": {
          "type": "cds.Decimal",
          "precision": 13,
          "scale": 3
        },
        "NET_ORDER_VALUE": {
          "type": "cds.Decimal",
          "precision": 38,
          "scale": 2
        },
        "AVG_PRICE_BY_MATERIAL": {
          "type": "cds.Decimal",
          "precision": 38,
          "scale": 2
        },
        "PRICE_DEVIATION": {
          "type": "cds.Double"
        },
        "PRICE_RATIO": {
          "type": "cds.Decimal",
          "precision": 38,
          "scale": 4
        },
        "ANOMALY_SCORE": {
          "type": "cds.Double"
        },
        "ANOMALY_FLAG": {
          "type": "cds.String"
        },
        "ANOMALY_DIRECTION": {
          "type": "cds.String"
        },
        "ANOMALY_CONFIDENCE": {
          "type": "cds.Decimal",
          "precision": 3,
          "scale": 0
        },
        "SCORED_AT": {
          "type": "cds.Timestamp"
        }
      }
    }
  }
}
$$
);